In [1]:
from pymongo import MongoClient
from collections import defaultdict
from datetime import datetime


In [6]:
# MongoDB connection
client = MongoClient("mongodb://localhost:27017/")
db = client["Retail_Business"]

orders = db["Orders"]
products_branch_wh = db["products_branch_warehouse"]  # WAREHOUSE layer


In [7]:
# Load all raw orders into memory
raw_orders = list(orders.find())

print(f"Total raw documents loaded: {len(raw_orders)}")


Total raw documents loaded: 200000


In [4]:
products_branch = defaultdict(lambda: {
    "branch_name": None,
    "product_name": None,
    "total_quantity": 0,
    "total_revenue": 0
})

In [8]:
for doc in raw_orders:
    key = (doc["branch"]["name"], doc["product"]["name"])

    row = products_branch[key]
    row["branch_name"] = doc["branch"]["name"]
    row["product_name"] = doc["product"]["name"]
    row["total_quantity"] += doc["product"]["quantity"]
    row["total_revenue"] += doc["sale"]["total_amount"]

In [9]:
products_branch_wh.delete_many({})
products_branch_wh.insert_many(list(products_branch.values()))

products_branch_wh.count_documents({})


197963